In [12]:
!nvidia-smi

Sun Sep 19 10:05:49 2021       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.63.01    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-SXM2...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   35C    P0    25W / 300W |      0MiB / 16160MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
pip install -U sentence-transformers

In [14]:
pip install transformers

In [15]:
pip install datasets

In [16]:
pip install tqdm

In [25]:
import torch
import os
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer
from transformers import pipeline
from tqdm.notebook import tqdm
from datasets import load_dataset,concatenate_datasets


DATASET_NAME ='imdb'
LABELED = False
MAX_LENGTH = 120


def prepare_dataset(name):
    # train_ds = load_dataset(name, split='train[:80%]')
    # val_ds = load_dataset(name, split='train[80%:]')
    train_ds = load_dataset(name, split='train')
    test_ds = load_dataset(name, split='test')

    # train_ds = train_ds.map(lambda examples: {'labels': examples['label']})
    # val_ds = val_ds.map(lambda examples: {'labels': examples['label']})
    # test_ds = test_ds.map(lambda examples: {'labels': examples['label']})

    # train_ds = train_ds.train_test_split(100)  # for experiment we only use 500 examples
    # train_ds = train_ds['test']

    test_ds = test_ds.train_test_split(1000)  # for experiment we only use 500 examples
    test_ds = test_ds['test']

    return train_ds,None,test_ds

In [18]:
embedder = SentenceTransformer('roberta-large')

train_ds, val_ds, test_ds = prepare_dataset(DATASET_NAME) # train_ds used as pool of examples, test_ds as the target dataset
class_names = test_ds.features["label"].names

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

if DATASET_NAME == 'imdb': 
  unmasker = pipeline('fill-mask', model='bert-base-cased', targets=['good', 'bad'])
elif DATASET_NAME == 'ag_news':
  unmasker = pipeline('fill-mask', model='bert-base-cased', targets=['World','Business','Sports','Science','Technology'])
  tokenizer.add_tokens('Sci/Tech')
else:
  raise Exception('Unsupported Dataset')

# prepocess the dataset
train_ds = train_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})
test_ds = test_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})


Downloading:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/899k [00:00<?, ?B/s]

Some weights of the model checkpoint at /root/.cache/torch/sentence_transformers/roberta-large were not used when initializing RobertaModel: ['lm_head.decoder.weight', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Downloading:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/436k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/25000 [00:00<?, ?ex/s]

  0%|          | 0/1000 [00:00<?, ?ex/s]

In [19]:
print(len(train_ds))
print(len(test_ds))

25000
1000


In [20]:
# preprocess examples in train_ds as form of e = (sentence, label, embedding)
examples = [{'text': e['text'], 'label': e['label'],
              'embedding': embedder.encode(e['text'], convert_to_tensor=False)} for e in tqdm(train_ds)]
corpus_embeddings = torch.FloatTensor([example['embedding'] for example in examples])


top_k = min(3, len(examples))

  0%|          | 0/25000 [00:00<?, ?it/s]

In [21]:
torch.save(corpus_embeddings,'corpus_embeddings.pt')

In [22]:
def label2name(label):
    if DATASET_NAME == 'imdb':
      if label == 1:
          return 'good'
      else:
          return 'bad'
    elif DATASET_NAME == 'ag_news':
      return class_names[label]
    else: 
      return 'null'


def generate_prompt(indices, query, Labeled=True):
    prompt = ''

    if Labeled:
      for id in indices:
        name = label2name(examples[id]['label'])
        if DATASET_NAME == 'imdb':
          prompt += examples[id]['text'] + ' The movie is ' + name + '.\n'
        elif DATASET_NAME == 'ag_news':
          if name == 'Sci/Tech':
            name = 'Science and Technology'
          prompt += examples[id]['text'] + ' The article is about ' + name + '.\n'
        else:
          raise Exception('Unsupported Dataset')
    else: # unlabeled priming
      for id in indices:
        if DATASET_NAME == 'imdb':
          tmp = examples[id]['text'] + ' The movie is [MASK].'
          prompt += unmasker(tmp)[0]['sequence'] +'\n '
        elif DATASET_NAME == 'ag_news':
          tmp = examples[id]['text'] + ' The article is about [MASK].'
          predicted = unmasker(tmp)[0]['token_str']
          if predicted == 'Science' or predicted == 'Technology':
            predicted = 'Science and Technology'
          prompt += examples[id]['text'] + ' The article is about ' + predicted + '.\n'
        else:
          raise Exception('Unsupported Dataset')
        
       
  
    if DATASET_NAME == 'imdb':
      prompt += query + ' The movie is [MASK].'
    elif DATASET_NAME == 'ag_news':
      prompt += query + ' The article is about [MASK].'
    else:
      raise Exception('Unsupported Dataset')

    return prompt

In [23]:

def predict(prompt):
    scores = unmasker(prompt)
    name = scores[0]['token_str']

    return name

In [26]:
# iterate over the whole test_dataset
correct_num = 0
total_num = 0
pbar = tqdm(test_ds, desc='Running')
corpus_embeddings = corpus_embeddings.to(device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
for query in pbar:
    query_embedding = embedder.encode(query['text'], convert_to_tensor=True)

    cos_scores = util.pytorch_cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    indices = top_results[1].tolist()
    indices.reverse()  # we put the more similar example, nearer to our query

    prompt = generate_prompt(indices, query['text'], Labeled=LABELED)
    

    if DATASET_NAME == 'imdb':
      true_name = 'good' if query['label'] ==1 else 'bad'
    elif  DATASET_NAME == 'ag_news':
      true_name = class_names[query['label']]
    else:
      raise Exception('Unsupported Dataset')

    predicted = predict(prompt)

    if true_name == 'Sci/Tech' and (predicted=='Science' or predicted=='Technology'):
      correct_num += 1
    elif predicted == true_name:
      correct_num += 1
    total_num += 1
    pbar.set_postfix({' binary_accuracy': correct_num/total_num})

print('%d/%d  accuracy: %.4f'%(correct_num, total_num, correct_num/total_num))

Running:   0%|          | 0/1000 [00:00<?, ?it/s]

557/1000  accuracy: 0.5570
